
# AI-Agent KYC Demo —

It demonstrates an agentic KYC pipeline using OpenRouter:

**Upload → OCR → Document Analysis → Identity Consistency → Face/Photo Review → Screening → Fraud Triage → Risk Engine → Orchestrator → Audit Case**

### Demo inputs

Upload three files:

1. **Identity document** — e.g. PAN / ID
2. **Address proof** — e.g. utility bill / bank statement
3. **Photo / selfie**

The notebook asks for each file separately, so **filenames do not matter**.



## What makes this more realistic?

Instead of asking one LLM:

> "Is this customer legitimate?"

we create separate agents with explicit responsibilities:

```text
                    ┌────────────────────┐
                    │   KYC Orchestrator  │
                    └─────────┬──────────┘
                              │
       ┌──────────────────────┼──────────────────────┐
       ▼                      ▼                      ▼
 Document Agent        Identity Agent          Photo Agent
 OCR + extraction      Cross-document          Photo quality /
                       consistency              comparison
       │                      │                      │
       └──────────────────────┼──────────────────────┘
                              ▼
                       Screening Agent
                       synthetic watchlist
                              │
                              ▼
                         Fraud Agent
                              │
                              ▼
                         Risk Engine
                              │
                    ┌─────────┴─────────┐
                    ▼                   ▼
              Auto workflow        Human review
```

The **LLM collects and explains evidence**. Deterministic checks and workflow rules control routing.


In [1]:
# 1. Install dependencies

!pip -q install requests pillow pandas ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 21.4 MB/s eta 0:00:00


In [2]:

# 2. Load OpenRouter secret

from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OpenRouter")

if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "OpenRouter secret not found. "
        "Open Colab Secrets, add a secret named 'OpenRouter', "
        "and grant this notebook access."
    )

print("✓ OpenRouter key loaded.")


✓ OpenRouter key loaded.


In [3]:
# 3. Imports and configuration

import base64
import json
import mimetypes
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from PIL import Image
from IPython.display import display

BASE_URL = "https://openrouter.ai/api/v1"

# OpenRouter currently exposes this model as the stable slug below.
VISION_MODEL = "openai/gpt-5.6-luna"
TEXT_MODEL = "openai/gpt-5.6-luna"

print("Vision model:", VISION_MODEL)
print("Text model:", TEXT_MODEL)

Vision model: openai/gpt-5.6-luna
Text model: openai/gpt-5.6-luna


## 4. OpenRouter API helpers

In [14]:
# 4. OpenRouter API helpers

def openrouter_headers():
    return {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "AI Agent KYC Walkthrough",
    }


def image_to_data_url(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Input image not found: {path}")

    # Verify that the uploaded file is a readable image before sending it.
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as exc:
        raise ValueError(f"Could not read image '{path.name}': {exc}") from exc

    suffix = path.suffix.lower()
    mime = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".jfif": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }.get(suffix)

    if mime is None:
        mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{encoded}"


def _message_text(response_json):
    """Normalize OpenRouter's message content to plain text."""
    content = response_json["choices"][0]["message"]["content"]
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and isinstance(item.get("text"), str):
                parts.append(item["text"])
        return "\n".join(parts)
    return str(content)


def _post_openrouter(payload, retries=3):
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.post(
                f"{BASE_URL}/chat/completions",
                headers=openrouter_headers(),
                json=payload,
                timeout=180,
            )

            if response.ok:
                return response.json()

            # Retry transient server/rate-limit errors; fail fast on auth/billing errors.
            if response.status_code not in {408, 429, 500, 502, 503, 504}:
                raise RuntimeError(
                    f"OpenRouter error {response.status_code}: {response.text[:2000]}"
                )

            last_error = RuntimeError(
                f"OpenRouter temporary error {response.status_code}: {response.text[:1000]}"
            )
        except requests.RequestException as exc:
            last_error = exc

        if attempt < retries - 1:
            time.sleep(2 ** attempt)

    raise RuntimeError(f"OpenRouter request failed after {retries} attempts: {last_error}")


def call_vision(prompt, image_paths, model=VISION_MODEL):
    image_paths = [Path(p) for p in image_paths]
    content = [{"type": "text", "text": prompt}]

    for p in image_paths:
        content.append({
            "type": "image_url",
            "image_url": {"url": image_to_data_url(p)},
        })

    payload = {
        "model": model,
        "temperature": 0,
        "max_tokens": 3000,
        "messages": [{"role": "user", "content": content}],
    }
    return _message_text(_post_openrouter(payload))


def call_text(prompt, model=TEXT_MODEL):
    payload = {
        "model": model,
        "temperature": 0,
        "max_tokens": 3000,
        "messages": [{"role": "user", "content": prompt}],
    }
    return _message_text(_post_openrouter(payload))


def extract_json(text):
    """Extract the first valid JSON object even if the model adds markdown fences."""
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    # First try the complete response.
    try:
        value = json.loads(text)
        if isinstance(value, dict):
            return value
    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder()
    for idx, char in enumerate(text):
        if char != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[idx:])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            continue

    raise ValueError("The model did not return a valid JSON object. Raw response:\n" + text[:4000])


print("✓ OpenRouter helpers ready.")

✓ OpenRouter helpers ready.



# 5. Upload the identity document

Upload the first document separately.

This avoids the filename-detection problem from the previous notebook.


In [11]:
# 5. Upload each demo document separately

from google.colab import files
from pathlib import Path
from PIL import Image

def upload_document(label):
    print(f"\nUpload your {label}:")
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise RuntimeError(
            f"Expected exactly 1 file for {label}, but received {len(uploaded)}. "
            "Please re-run this cell."
        )

    filename = next(iter(uploaded))
    return Path(filename)


# 1) Identity document
identity_path = upload_document("Identity document (PAN / passport / ID)")

# 2) Address proof
address_path = upload_document("Address proof")

# 3) Photo / selfie
photo_path = upload_document("Photo / selfie")


selected_files = {
    "identity_document": identity_path,
    "address_proof": address_path,
    "photo": photo_path,
}


# Validate that all uploaded files exist and are readable images
for label, path in selected_files.items():
    if not path.exists():
        raise FileNotFoundError(f"{label} file not found: {path}")

    try:
        with Image.open(path) as img:
            print(
                f"✓ {label}: {path.name} | "
                f"{img.size[0]}x{img.size[1]} | {img.format}"
            )
    except Exception as exc:
        raise ValueError(
            f"{label} is not a readable image: {path}"
        ) from exc

print("\n✓ All three documents uploaded successfully.")


Upload your Identity document (PAN / passport / ID):


Saving pan.jfif to pan.jfif

Upload your Address proof:


Saving demo_address_proof.png to demo_address_proof.png

Upload your Photo / selfie:


Saving synthetic_portrait.png to synthetic_portrait.png
✓ identity_document: pan.jfif | 2592x1632 | JPEG
✓ address_proof: demo_address_proof.png | 786x559 | PNG
✓ photo: synthetic_portrait.png | 399x501 | PNG

✓ All three documents uploaded successfully.


## 6. Upload the address proof

## 7. Upload the photo / selfie


# 8. Customer-declared information

For a real KYC application this would normally come from the onboarding form.

For this walkthrough, enter the information the applicant claims is theirs.

The agents will compare this against the documents.


In [12]:
# Preview the exact files that will be sent to OpenRouter

for label, path in selected_files.items():
    print(f"\n--- {label}: {path.name} ---")
    display(Image.open(path).copy())

Output hidden; open in https://colab.research.google.com to view.

In [16]:

CUSTOMER_DECLARED = {
    "customer_id": "DEMO-" + uuid.uuid4().hex[:8].upper(),

    # Change these values to demonstrate different mismatch scenarios.
    "name": "Twitterpreet Singh",
    "date_of_birth": "14/05/1995",
    "address": "10 Downing St, Westminster, London SW1A 2AA, UK",

    # Optional fields:
    "declared_document_number": "BWZPS1234R",
}

print(json.dumps(CUSTOMER_DECLARED, indent=2))


{
  "customer_id": "DEMO-D915E9CC",
  "name": "Twitterpreet Singh",
  "date_of_birth": "14/05/1995",
  "address": "10 Downing St, Westminster, London SW1A 2AA, UK",
  "declared_document_number": "BWZPS1234R"
}


# 9. Agent 1 — Document Intelligence / OCR

In [17]:

DOCUMENT_AGENT_PROMPT = '''
You are the Document Intelligence Agent in a KYC workflow.

You are given two images:
1. identity_document
2. address_proof

Extract ONLY information visibly present in the images.

Return ONLY valid JSON:

{
  "identity_document": {
    "document_type": "",
    "issuer": "",
    "name": "",
    "date_of_birth": "",
    "document_number": "",
    "address": "",
    "expiry_date": "",
    "visible_warnings_or_labels": [],
    "image_quality": "GOOD|FAIR|POOR",
    "extraction_confidence": 0.0
  },
  "address_proof": {
    "document_type": "",
    "issuer": "",
    "name": "",
    "date_of_birth": "",
    "document_number": "",
    "address": "",
    "visible_warnings_or_labels": [],
    "image_quality": "GOOD|FAIR|POOR",
    "extraction_confidence": 0.0
  },
  "document_observations": []
}

Rules:
- Never invent missing fields.
- Preserve exact spelling where readable.
- Report uncertainty rather than guessing.
- Report visible demo/test labels.
- Do not claim government authenticity or legal validity.
'''

ocr_raw = call_vision(
    DOCUMENT_AGENT_PROMPT,
    [identity_path, address_path],
)

ocr_result = extract_json(ocr_raw)

print(json.dumps(ocr_result, indent=2, ensure_ascii=False))


{
  "identity_document": {
    "document_type": "Permanent Account Number",
    "issuer": "INCOME TAX DEPARTMENT; GOVT. OF INDIA",
    "name": "TWITTERPREET SINGH",
    "date_of_birth": "14/05/1995",
    "document_number": "BWZPS1234R",
    "address": "",
    "expiry_date": "",
    "visible_warnings_or_labels": [],
    "image_quality": "GOOD",
    "extraction_confidence": 0.99
  },
  "address_proof": {
    "document_type": "Smart Card",
    "issuer": "ShuftiPro",
    "name": "John Livone",
    "date_of_birth": "06-09-1986",
    "document_number": "A123456",
    "address": "10 Downing st, Westminster, London SW1A 2AA, UK",
    "expiry_date": "12-11-2030",
    "visible_warnings_or_labels": [
      "Real Identity",
      "Test Card"
    ],
    "image_quality": "GOOD",
    "extraction_confidence": 0.98
  },
  "document_observations": [
    "The names and dates of birth shown on the two documents do not match.",
    "The address proof visibly contains the label \"Test Card\"."
  ]
}


### OCR result

In [18]:

def flatten_ocr(result):
    rows = []

    for doc_name, data in result.items():
        if isinstance(data, dict):
            for key, value in data.items():
                if isinstance(value, (dict, list)):
                    value = json.dumps(value, ensure_ascii=False)
                rows.append({
                    "document": doc_name,
                    "field": key,
                    "value": value,
                })

    return pd.DataFrame(rows)


display(flatten_ocr(ocr_result))


,document,field,value
0,identity_document,document_type,Permanent Account Number
1,identity_document,issuer,INCOME TAX DEPARTMENT; GOVT. OF INDIA
2,identity_document,name,TWITTERPREET SINGH
3,identity_document,date_of_birth,14/05/1995
4,identity_document,document_number,BWZPS1234R
5,identity_document,address,
6,identity_document,expiry_date,
7,identity_document,visible_warnings_or_labels,[]
8,identity_document,image_quality,GOOD
9,identity_document,extraction_confidence,0.99


# 10. Agent 2 — Identity Consistency

In [19]:

IDENTITY_AGENT_PROMPT = '''
You are the Identity Consistency Agent.

Compare:
A. customer-declared information
B. extracted identity-document information
C. extracted address-proof information

Return ONLY valid JSON:

{
  "name": {
    "status": "MATCH|MISMATCH|UNCLEAR",
    "details": ""
  },
  "date_of_birth": {
    "status": "MATCH|MISMATCH|UNCLEAR",
    "details": ""
  },
  "document_number": {
    "status": "MATCH|MISMATCH|UNCLEAR",
    "details": ""
  },
  "address": {
    "status": "MATCH|MISMATCH|UNCLEAR|NOT_AVAILABLE",
    "details": ""
  },
  "cross_document_identity": "CONSISTENT|INCONSISTENT|UNCLEAR",
  "findings": []
}

Important:
- Compare text, do not infer identity from appearance.
- Do not claim legal identity verification.
- Do not decide whether a document is genuine.
- Clearly distinguish an observed mismatch from a fraud conclusion.
'''

identity_prompt = (
    IDENTITY_AGENT_PROMPT
    + "\n\nCUSTOMER DECLARED:\n"
    + json.dumps(CUSTOMER_DECLARED, indent=2)
    + "\n\nDOCUMENT OCR:\n"
    + json.dumps(ocr_result, indent=2)
)

identity_raw = call_text(identity_prompt)
identity_result = extract_json(identity_raw)

print(json.dumps(identity_result, indent=2, ensure_ascii=False))


{
  "name": {
    "status": "MISMATCH",
    "details": "Customer-declared name matches the identity document ('Twitterpreet Singh' / 'TWITTERPREET SINGH') but differs from the address-proof name ('John Livone')."
  },
  "date_of_birth": {
    "status": "MISMATCH",
    "details": "Customer-declared date of birth matches the identity document ('14/05/1995') but differs from the address-proof date of birth ('06-09-1986')."
  },
  "document_number": {
    "status": "MATCH",
    "details": "The declared document number 'BWZPS1234R' matches the identity-document number. The address proof has a separate document number, 'A123456'."
  },
  "address": {
    "status": "MATCH",
    "details": "The declared address matches the address shown on the address proof, allowing for capitalization differences."
  },
  "cross_document_identity": "INCONSISTENT",
  "findings": [
    "The address proof name differs from the customer-declared name and identity-document name.",
    "The address proof date of bi

# 11. Agent 3 — Photo / Document Face Consistency

In [20]:

PHOTO_AGENT_PROMPT = '''
You are a visual KYC review agent.

You receive:
1. identity document image
2. selfie/photo image

Assess only what can be visually observed.

Return ONLY valid JSON:

{
  "photo_quality": "GOOD|FAIR|POOR",
  "identity_document_photo_present": true,
  "selfie_face_present": true,
  "visual_similarity": "HIGH|MEDIUM|LOW|UNCLEAR",
  "face_comparison_confidence": 0.0,
  "possible_mismatch": true,
  "observations": []
}

Strict rules:
- Do NOT identify or name the person.
- Do NOT infer sensitive traits.
- Do NOT claim biometric verification.
- Do NOT claim liveness.
- This is a demo-level visual consistency assessment.
- If either image is unsuitable, use UNCLEAR.
'''

photo_raw = call_vision(
    PHOTO_AGENT_PROMPT,
    [identity_path, photo_path],
)

photo_result = extract_json(photo_raw)

print(json.dumps(photo_result, indent=2, ensure_ascii=False))


{
  "photo_quality": "GOOD",
  "identity_document_photo_present": true,
  "selfie_face_present": true,
  "visual_similarity": "LOW",
  "face_comparison_confidence": 0.97,
  "possible_mismatch": true,
  "observations": [
    "Both faces are clearly visible and sufficiently well-lit for visual comparison.",
    "The document photo shows a person wearing a turban with substantial facial hair, while the selfie shows short hair and no visible facial hair.",
    "Visible facial appearance and styling differ substantially between the two images."
  ]
}


# 12. Agent 4 — Document Risk / Fraud Triage

In [21]:

FRAUD_AGENT_PROMPT = '''
You are a KYC fraud-triage agent.

Use the OCR, identity-consistency, and photo-analysis outputs.

Return ONLY valid JSON:

{
  "document_tampering_risk": 0.0,
  "identity_inconsistency_risk": 0.0,
  "photo_mismatch_risk": 0.0,
  "synthetic_document_risk": 0.0,
  "overall_triage": "LOW|MEDIUM|HIGH|UNCLEAR",
  "reasons": []
}

Rules:
- These are triage signals, not a fraud finding.
- A mismatch is evidence requiring review, not proof of fraud.
- Do not claim deepfake detection.
- Do not claim biometric verification.
- Do not invent forensic evidence that is not present in the inputs.
'''

fraud_prompt = (
    FRAUD_AGENT_PROMPT
    + "\n\nOCR:\n"
    + json.dumps(ocr_result, indent=2)
    + "\n\nIDENTITY ANALYSIS:\n"
    + json.dumps(identity_result, indent=2)
    + "\n\nPHOTO ANALYSIS:\n"
    + json.dumps(photo_result, indent=2)
)

fraud_raw = call_text(fraud_prompt)
fraud_result = extract_json(fraud_raw)

print(json.dumps(fraud_result, indent=2, ensure_ascii=False))


{
  "document_tampering_risk": 0.15,
  "identity_inconsistency_risk": 0.98,
  "photo_mismatch_risk": 0.97,
  "synthetic_document_risk": 0.9,
  "overall_triage": "HIGH",
  "reasons": [
    "The address proof name and date of birth do not match the customer-declared identity or the identity document.",
    "The address proof visibly contains the labels \"Real Identity\" and \"Test Card\".",
    "The photo analysis reports low visual similarity between the identity-document photo and the selfie, with substantially different visible facial appearance and styling.",
    "The identity document itself has good image quality and consistent declared identity details; no direct tampering evidence was provided."
  ]
}


# 13. Agent 5 — Screening

In [22]:

# Synthetic screening data for the demo.
# Replace this function with your approved screening provider in production.

SYNTHETIC_WATCHLIST = [
    {
        "name": "JOHN DOE",
        "category": "SANCTIONS",
        "record_id": "DEMO-SAN-001",
    },
    {
        "name": "ALEX MORGAN",
        "category": "PEP",
        "record_id": "DEMO-PEP-001",
    },
]


def normalize_name(value):
    return re.sub(r"\s+", " ", str(value or "").upper().strip())


def screen_name(name):
    normalized = normalize_name(name)
    if not normalized:
        return {
            "status": "UNCLEAR",
            "matches": [],
            "source": "Synthetic demo screening list",
            "reason": "No usable name was extracted from the identity document.",
        }

    matches = [
        record for record in SYNTHETIC_WATCHLIST
        if normalize_name(record["name"]) == normalized
    ]

    return {
        "status": "POTENTIAL_MATCH" if matches else "CLEAR_DEMO_LIST",
        "matches": matches,
        "source": "Synthetic demo screening list",
    }


screening_result = screen_name(
    ocr_result["identity_document"].get("name")
)

print(json.dumps(screening_result, indent=2))


{
  "status": "CLEAR_DEMO_LIST",
  "matches": [],
  "source": "Synthetic demo screening list"
}


# 14. Deterministic Risk Engine

In [23]:
def risk_engine(identity, photo, fraud, screening):
    score = 0.0
    reasons = []

    # Identity mismatches
    weights = {
        "name": 0.25,
        "date_of_birth": 0.20,
        "document_number": 0.20,
        "address": 0.15,
    }
    for field, weight in weights.items():
        status = identity.get(field, {}).get("status")
        if status == "MISMATCH":
            score += weight
            reasons.append(f"{field.replace('_', ' ').title()} mismatch")
        elif status in {"UNCLEAR", "NOT_AVAILABLE"}:
            reasons.append(f"{field.replace('_', ' ').title()} could not be confidently compared")

    # Screening
    if screening.get("matches"):
        score += 0.50
        reasons.append("Potential screening match")

    # Vision/fraud triage signals
    score += 0.20 * float(fraud.get("identity_inconsistency_risk", 0) or 0)
    score += 0.20 * float(fraud.get("photo_mismatch_risk", 0) or 0)
    score += 0.10 * float(fraud.get("document_tampering_risk", 0) or 0)
    score += 0.10 * float(fraud.get("synthetic_document_risk", 0) or 0)

    score = min(score, 1.0)

    if score < 0.25:
        band = "LOW"
    elif score < 0.60:
        band = "MEDIUM"
    else:
        band = "HIGH"

    return {
        "risk_score": round(score, 3),
        "risk_band": band,
        "reasons": reasons,
    }

risk_result = risk_engine(
    identity_result,
    photo_result,
    fraud_result,
    screening_result,
)

print(json.dumps(risk_result, indent=2))

{
  "risk_score": 0.945,
  "risk_band": "HIGH",
  "reasons": [
    "Name mismatch",
    "Date Of Birth mismatch"
  ]
}


# 15. Orchestrator Agent

In [24]:

ORCHESTRATOR_PROMPT = '''
You are the KYC workflow orchestrator.

Choose exactly one next workflow state:

- STRAIGHT_THROUGH_PROCESSING
- COLLECT_MISSING_INFORMATION
- HUMAN_REVIEW
- ENHANCED_DUE_DILIGENCE

Return ONLY JSON:

{
  "next_state": "",
  "reason": "",
  "required_actions": []
}

Rules:
- A potential sanctions/PEP match must route to HUMAN_REVIEW.
- Material identity inconsistency should route to HUMAN_REVIEW.
- High fraud-triage risk should route to HUMAN_REVIEW or ENHANCED_DUE_DILIGENCE.
- Missing evidence should route to COLLECT_MISSING_INFORMATION.
- Do not claim that an LLM has legally verified identity.
- Do not make a real-world approval decision.
'''

orchestrator_input = {
    "customer": CUSTOMER_DECLARED,
    "identity": identity_result,
    "photo": photo_result,
    "fraud": fraud_result,
    "screening": screening_result,
    "risk": risk_result,
}

orchestrator_raw = call_text(
    ORCHESTRATOR_PROMPT
    + "\n\nEVIDENCE:\n"
    + json.dumps(orchestrator_input, indent=2)
)

orchestrator_result = extract_json(orchestrator_raw)

print(json.dumps(orchestrator_result, indent=2, ensure_ascii=False))


{
  "next_state": "HUMAN_REVIEW",
  "reason": "Material cross-document identity inconsistencies, substantial selfie-to-document photo mismatch, and high fraud-triage risk require manual assessment.",
  "required_actions": [
    "Review the identity document, address proof, and selfie comparison manually.",
    "Investigate the address proof name and date-of-birth discrepancies and the labels \"Real Identity\" and \"Test Card\".",
    "Determine whether additional reliable identity and address evidence is required."
  ]
}


# 16. Analyst Case View

In [25]:

case_id = "CASE-" + uuid.uuid4().hex[:8].upper()

case = {
    "case_id": case_id,
    "customer_id": CUSTOMER_DECLARED["customer_id"],
    "customer_declared": CUSTOMER_DECLARED,
    "ocr": ocr_result,
    "identity_consistency": identity_result,
    "photo_analysis": photo_result,
    "fraud_triage": fraud_result,
    "screening": screening_result,
    "risk": risk_result,
    "orchestrator": orchestrator_result,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "demo_only": True,
}

print(json.dumps(case, indent=2, ensure_ascii=False))


{
  "case_id": "CASE-BD34F7A1",
  "customer_id": "DEMO-D915E9CC",
  "customer_declared": {
    "customer_id": "DEMO-D915E9CC",
    "name": "Twitterpreet Singh",
    "date_of_birth": "14/05/1995",
    "address": "10 Downing St, Westminster, London SW1A 2AA, UK",
    "declared_document_number": "BWZPS1234R"
  },
  "ocr": {
    "identity_document": {
      "document_type": "Permanent Account Number",
      "issuer": "INCOME TAX DEPARTMENT; GOVT. OF INDIA",
      "name": "TWITTERPREET SINGH",
      "date_of_birth": "14/05/1995",
      "document_number": "BWZPS1234R",
      "address": "",
      "expiry_date": "",
      "visible_warnings_or_labels": [],
      "image_quality": "GOOD",
      "extraction_confidence": 0.99
    },
    "address_proof": {
      "document_type": "Smart Card",
      "issuer": "ShuftiPro",
      "name": "John Livone",
      "date_of_birth": "06-09-1986",
      "document_number": "A123456",
      "address": "10 Downing st, Westminster, London SW1A 2AA, UK",
      "expi

In [26]:

# Compact analyst dashboard

dashboard = pd.DataFrame([
    ["Document OCR", "COMPLETED"],
    ["Identity consistency", identity_result["cross_document_identity"]],
    ["Photo assessment", photo_result["visual_similarity"]],
    ["Screening", screening_result["status"]],
    ["Fraud triage", fraud_result["overall_triage"]],
    ["Risk band", risk_result["risk_band"]],
    ["Workflow", orchestrator_result["next_state"]],
], columns=["Check", "Result"])

display(dashboard)


,Check,Result
0,Document OCR,COMPLETED
1,Identity consistency,INCONSISTENT
2,Photo assessment,LOW
3,Screening,CLEAR_DEMO_LIST
4,Fraud triage,HIGH
5,Risk band,HIGH
6,Workflow,HUMAN_REVIEW


# 17. Audit Trail

In [27]:

audit_events = []

def audit(agent, event, status, details):
    audit_events.append({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "agent": agent,
        "event": event,
        "status": status,
        "details": details,
    })


audit("Document Agent", "OCR and extraction", "COMPLETED", ocr_result)
audit("Identity Agent", "Cross-document comparison", identity_result["cross_document_identity"], identity_result)
audit("Photo Agent", "Visual consistency", photo_result["visual_similarity"], photo_result)
audit("Screening Agent", "Demo screening", screening_result["status"], screening_result)
audit("Fraud Agent", "Fraud triage", fraud_result["overall_triage"], fraud_result)
audit("Risk Engine", "Risk calculation", risk_result["risk_band"], risk_result)
audit("Orchestrator", "Workflow routing", orchestrator_result["next_state"], orchestrator_result)

audit_df = pd.DataFrame(audit_events)

display(
    audit_df[
        ["timestamp", "agent", "event", "status"]
    ]
)


,timestamp,agent,event,status
0,2026-09-18T09:48:41.281762+00:00,Document Agent,OCR and extraction,COMPLETED
1,2026-09-18T09:48:41.281885+00:00,Identity Agent,Cross-document comparison,INCONSISTENT
2,2026-09-18T09:48:41.281974+00:00,Photo Agent,Visual consistency,LOW
3,2026-09-18T09:48:41.282048+00:00,Screening Agent,Demo screening,CLEAR_DEMO_LIST
4,2026-09-18T09:48:41.282117+00:00,Fraud Agent,Fraud triage,HIGH
5,2026-09-18T09:48:41.282183+00:00,Risk Engine,Risk calculation,HIGH
6,2026-09-18T09:48:41.282249+00:00,Orchestrator,Workflow routing,HUMAN_REVIEW



# 18. Demo: create a deliberate mismatch

This is useful for the walkthrough.

Change:

```python
CUSTOMER_DECLARED["name"] = "Someone Else"
```

or change the declared DOB/address, then **rerun cells 9–17**.

The identity agent should surface the discrepancy, the fraud-triage agent should treat it as a signal rather than proof of fraud, the risk engine should increase the score, and the orchestrator can route the case to human review.

This is a better demonstration of an agentic KYC system than showing only a successful onboarding path.

---

# 19. Production architecture

The notebook maps to a production architecture like this:

```text
KYC Web / Mobile App
        |
        v
API Gateway
        |
        v
Workflow Orchestrator
        |
        +---- Document Intelligence
        |       |
        |       +---- OCR
        |       +---- Document classification
        |       +---- Authenticity service
        |
        +---- Identity Verification
        |       |
        |       +---- ID database/API
        |       +---- Name/DOB/address matching
        |
        +---- Biometric Verification
        |       |
        |       +---- Face match
        |       +---- Liveness
        |
        +---- AML Screening
        |       |
        |       +---- Sanctions
        |       +---- PEP
        |       +---- Adverse media
        |
        +---- Fraud
        |       |
        |       +---- Device intelligence
        |       +---- Velocity
        |       +---- Synthetic identity
        |       +---- Deepfake/document forensics
        |
        v
Policy / Risk Engine
        |
   +----+----+
   |         |
   v         v
Straight   Human
through    review
   |         |
   +----+----+
        |
        v
Case + Immutable Audit Trail
```

### Key design principle

**Do not give one LLM unrestricted authority to approve or reject customers.**

Use AI for:
- extraction
- normalization
- evidence correlation
- anomaly explanation
- case summarization
- workflow routing

Use deterministic controls / specialized services for:
- authoritative verification
- biometric verification
- sanctions screening
- policy rules
- auditability

And keep human review for cases your compliance policy requires.

---
